# Homework 3 Solutions

In [1]:
# This is a code cell that imports the necessary libraries for our session.
import numpy as np                        # NumPy for numerical computations
import scipy as sp                        # SciPy for scientific computing
import sympy as sym                       # SymPy for symbolic mathematics
import matplotlib as mpl                  # Matplotlib for plotting
import matplotlib.pyplot as plt           # Matplotlib pyplot interface
from scipy.integrate import solve_ivp     # ODE solver
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

This page collects full solutions to select exercises from Homework 3.
Each solution is worked out “by hand” and then, where helpful, confirmed
symbolically using SymPy or other computational tools. See the [Homework
Solutions](https://math341diffeq.netlify.app/select_hw_solutions) page
for the full list.

## Section 1.5.1 — Autonomous Systems

### Exercise 2(b)

For the model $x' = x(4-x)(5-x)^2$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = x(4-x)(5-x)^2 = 0$ gives three equilibria:
$x = 0,\ 4,\ 5$.

Theorem 1.32 classifies a *hyperbolic* equilibrium (one where
$f'(x_{eq}) \neq 0$) by the sign of $f'(x_{eq})$: negative means stable,
positive means unstable. Differentiating and evaluating, $$
f'(0) = 100 > 0 \ (\text{unstable}), \qquad f'(4) = -4 < 0 \ (\text{stable}), \qquad f'(5) = 0 \ (\text{inconclusive}).
$$

Since $f'(5) = 0$, Theorem 1.32 does not directly apply at $x=5$ and we
examine the sign of $f(x)$ directly instead. Because $(5-x)^2 \geq 0$
never changes sign, the sign of $f(x)$ near $x=5$ is controlled by
$x(4-x)$, which is negative for $x$ near $5$ on **both** sides.
Tabulating the sign of $f$ on each interval:

| Interval       | $(-\infty, 0)$ | $(0,4)$ | $(4,5)$ | $(5,\infty)$ |
|----------------|----------------|---------|---------|--------------|
| sign of $f(x)$ | $-$            | $+$     | $-$     | $-$          |

- At $x=0$: sign goes $- \to +$, so trajectories are pushed away on both
  sides $\Rightarrow$ **unstable**.
- At $x=4$: sign goes $+ \to -$, so trajectories are pulled in from both
  sides $\Rightarrow$ **stable**.
- At $x=5$: the sign is negative on both sides, so trajectories from
  below move *away* from $5$ (down toward $4$) while trajectories from
  above move *toward* $5$ $\Rightarrow$ **semistable**.

In [2]:
def f2b(x):
    return x*(4-x)*(5-x)**2

xmin, xmax = -1, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2b(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)          # unstable
ax1.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)             # stable
ax1.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)                          # semistable
ax1.set_ylim(-60, 60)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = x(4-x)(5-x)^2$")

bounds = [xmin, 0, 4, 5, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2b(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [3]:
x = sym.symbols('x')
f = x*(4-x)*(5-x)**2

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [0, 4, 5]
  f'(0) = 100
  f'(4) = -4
  f'(5) = 0

> **Note**
>
> SymPy confirms the equilibria $x = 0, 4, 5$ and the derivative values
> used above; the vanishing derivative at $x=5$ is exactly what forced
> the direct sign analysis.

**(c) — Representative time series.**

In [4]:
def f2b_t(t, x):
    return x*(4-x)*(5-x)**2

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [-0.5, 0.5, 1, 3, 4.5, 5, 5.5, 6]:
    sol = solve_ivp(f2b_t, [0, 3], [x0], dense_output=True, max_step=0.01, rtol=1e-8)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [0, 4, 5]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(-3, 7)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = x(4-x)(5-x)^2$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

### Exercise 2(g)

For the model $x' = (4-x)(2-x)^3$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = (4-x)(2-x)^3 = 0$ gives two equilibria: $x = 2$ (a
triple root of the factor $(2-x)^3$) and $x = 4$.

$$
f'(2) = 0 \ (\text{inconclusive}), \qquad f'(4) = 8 > 0 \ (\text{unstable}).
$$

Since $f'(2)=0$ (the triple root makes $x=2$ non-hyperbolic), we again
check signs directly:

| Interval       | $(-\infty, 2)$ | $(2,4)$ | $(4,\infty)$ |
|----------------|----------------|---------|--------------|
| sign of $f(x)$ | $+$            | $-$     | $+$          |

- At $x=2$: sign goes $+ \to -$ $\Rightarrow$ **stable**.
- At $x=4$: sign goes $- \to +$, consistent with $f'(4)>0$ $\Rightarrow$
  **unstable**.

In [5]:
def f2g(x):
    return (4-x)*(2-x)**3

xmin, xmax = 0, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2g(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)     # stable
ax1.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5) # unstable
ax1.set_ylim(-30, 30)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = (4-x)(2-x)^3$")

bounds = [xmin, 2, 4, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2g(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [6]:
x = sym.symbols('x')
f = (4-x)*(2-x)**3

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [2, 4]
  f'(2) = 0
  f'(4) = 8

**(c) — Representative time series.**

In [7]:
def f2g_t(t, x):
    return (4-x)*(2-x)**3

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [0.5, 1.5, 2, 3, 3.9, 4, 4.1, 4.5]:
    sol = solve_ivp(f2g_t, [0, 2], [x0], dense_output=True, max_step=0.005, rtol=1e-9)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [2, 4]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(0, 8)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = (4-x)(2-x)^3$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> **Tip**
>
> Notice how flat the curves are near $x=2$ compared with how sharply
> solutions bend away near $x=4$. This reflects $f'(2)=0$ versus
> $f'(4) = 8$: convergence toward a non-hyperbolic equilibrium (a
> repeated root) is much slower than the exponential rate predicted by
> Theorem 1.32 at a hyperbolic one.

------------------------------------------------------------------------

### Exercise 3(c)

Use analytical or graphical methods to determine the equilibria of
$R' = \dfrac{3R}{1+R^2} - 1$.

**By Hand.**

Setting $f(R) = \dfrac{3R}{1+R^2} - 1 = 0$: $$
\frac{3R}{1+R^2} = 1 \quad \Longrightarrow \quad 3R = 1 + R^2 \quad \Longrightarrow \quad R^2 - 3R + 1 = 0.
$$

By the quadratic formula, $$
R = \frac{3 \pm \sqrt{9-4}}{2} = \frac{3 \pm \sqrt{5}}{2}.
$$

$$
\boxed{R_1 = \frac{3-\sqrt{5}}{2} \approx 0.382, \qquad R_2 = \frac{3+\sqrt{5}}{2} \approx 2.618}
$$

Graphically, these are exactly the two places where the curve $f(R)$
crosses the $R$-axis:

In [8]:
def f3c(R):
    return 3*R/(1+R**2) - 1

R = np.linspace(-1, 6, 500)
R1 = (3 - np.sqrt(5)) / 2
R2 = (3 + np.sqrt(5)) / 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(R, f3c(R), color='steelblue', lw=2)
ax.axhline(0, color='black', lw=1)
ax.plot(R1, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax.plot(R2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax.annotate(f'$R_1 \\approx {R1:.3f}$', xy=(R1, 0), xytext=(R1-0.3, 0.6),
            fontsize=9, ha='right', arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate(f'$R_2 \\approx {R2:.3f}$', xy=(R2, 0), xytext=(R2+0.3, 0.6),
            fontsize=9, ha='left', arrowprops=dict(arrowstyle='->', lw=1))
ax.set_xlabel('$R$')
ax.set_ylabel("$R' = f(R)$")
ax.set_title(r"Growth rate $f(R) = \dfrac{3R}{1+R^2} - 1$")
plt.tight_layout()
plt.show()

In [9]:
R = sym.symbols('R')
f = 3*R/(1+R**2) - 1
eqs = sym.solve(sym.Eq(f, 0), R)
print("Equilibria (exact):", eqs)
print("Equilibria (decimal):", [sym.N(e) for e in eqs])

Equilibria (exact): [3/2 - sqrt(5)/2, sqrt(5)/2 + 3/2]
Equilibria (decimal): [0.381966011250105, 2.61803398874989]

> **Tip**
>
> Although the exercise only asks for the equilibria, stability comes
> almost for free from the same plot: the curve crosses from $+$ to $-$
> at $R_2$ (stable) and from $-$ to $+$ at $R_1$ (unstable) — consistent
> with $f'(R_1) \approx 1.95 > 0$ and $f'(R_2) \approx -0.28 < 0$.

------------------------------------------------------------------------

### Exercise 6 (Heat transfer)

Heat transfer by radiation from a body to its surroundings is modeled by
the Stefan–Boltzmann law $$
\frac{dT}{dt} = -k\left(T^4 - S^4\right),
$$ where $T = T(t)$ is the absolute temperature of the body and $S$ is
the (constant) temperature of the surroundings.

**(a) Phase line diagram and nature of the solution.**

The only equilibrium (for $T, S \geq 0$) is $T = S$, since $T^4 = S^4$
with $T,S\ge 0$ forces $T=S$. Writing $f(T) = -k(T^4-S^4)$:

- For $T > S$: $T^4 > S^4$, so $f(T) < 0$ (temperature decreasing — the
  body cools toward $S$).
- For $T < S$: $T^4 < S^4$, so $f(T) > 0$ (temperature increasing — the
  body warms toward $S$).

So $T = S$ is a **stable** equilibrium, and every solution approaches it
*monotonically* (no overshoot or oscillation) — exactly the physically
expected behavior of a body radiating heat until it reaches thermal
equilibrium with its surroundings.

In [10]:
k_demo = 2.0e-12
S_demo = 300

def fT(T, S):
    return -k_demo*(T**4 - S**4)

Tvals = np.linspace(0, 2*S_demo, 300)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.5),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax1.plot(Tvals, fT(Tvals, S_demo), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax1.set_ylabel("$T' = -k(T^4-S^4)$")
ax1.set_title("Growth rate for $T' = -k(T^4 - S^4)$")

bounds = [0, S_demo, 2*S_demo]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(fT(mid, S_demo))
    x0, x1 = (a + 0.05*S_demo, b - 0.05*S_demo) if sign > 0 else (b - 0.05*S_demo, a + 0.05*S_demo)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$T$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

**(b) Approximate solution neglecting $S$.**

With $T(0) = 2000\,\text{K}$ and $S = 300\,\text{K}$, since $S \ll T$ we
drop the $S^4$ term and solve $$
\frac{dT}{dt} = -kT^4, \qquad k = 2.0\times10^{-12}\ \text{K}^{-3}\text{sec}^{-1}.
$$

This is separable: $$
\frac{dT}{T^4} = -k\,dt \quad \Longrightarrow \quad -\frac{1}{3}T^{-3} = -kt + C \quad \Longrightarrow \quad T^{-3} = 3kt + T_0^{-3}.
$$

Solving for $T$ and substituting $T_0 = 2000$: $$
T(t) = \frac{T_0}{\left(1 + 3kT_0^3\,t\right)^{1/3}}.
$$

Since $3kT_0^3 = 3(2.0\times10^{-12})(2000)^3 = 0.048\ \text{sec}^{-1}$,

$$
\boxed{T(t) = \frac{2000}{(1+0.048\,t)^{1/3}} \ \text{K}, \qquad t \text{ in seconds}.}
$$

In [11]:
t = sym.symbols('t', positive=True)
T = sym.Function('T')
k_val = sym.Rational(2, 10**12)

eq = sym.Eq(sym.diff(T(t), t), -k_val*T(t)**4)
sol = sym.dsolve(eq, T(t), ics={T(0): 2000})
print("Solution:", sym.simplify(sol))
print("Coefficient 3*k*T0^3 =", sym.N(3*k_val*2000**3))

Solution: Eq(T(t), 10000/(6*t + 125)**(1/3))
Coefficient 3*k*T0^3 = 0.0480000000000000

> **Note**
>
> SymPy confirms
> $T(t)^{-3} = \dfrac{3t}{5\times10^{11}} + \dfrac{1}{8\times 10^9}$ (an
> equivalent way of writing the boxed formula), and the numerically
> computed coefficient matches $0.048$.

In [12]:
T0 = 2000
k = 2.0e-12
S = 300

t_vals = np.linspace(0, 3000, 400)
T_approx = T0 * (1 + 3*k*T0**3*t_vals)**(-1/3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, T_approx, color='steelblue', lw=2, label=r'$T(t) = 2000(1+0.048t)^{-1/3}$')
ax.axhline(S, color='gray', linestyle='--', lw=1, label='$S = 300$ K')
ax.set_xlabel('$t$ (seconds)')
ax.set_ylabel('$T$ (K)')
ax.set_title('Approximate cooling curve, neglecting $S$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Tip**
>
> The approximate solution is only valid while $T \gg S$. By
> $t \approx 3000$ sec the body has cooled to around $380\,$K — close
> enough to $S = 300\,$K that neglecting $S^4$ in the original equation
> is no longer well justified; the full equation (including $S^4$) would
> need to be solved (numerically) to accurately track the final approach
> to equilibrium.

------------------------------------------------------------------------

## Section 1.5.2 — Bifurcations

For each model below (containing a parameter $h$): (a) find the
equilibria in terms of $h$ and determine their stability using Theorem
1.32, to the extent possible; (b) construct a bifurcation diagram
showing how the equilibria depend on $h$, labeling the branches as
stable or unstable.

### Exercise 1(a): $x' = h - x^3$

**By Hand.**

Setting $f(x) = h - x^3 = 0$ gives the unique real equilibrium $$
x_{eq}(h) = h^{1/3}
$$ (the cube root function is one-to-one on $\mathbb{R}$, so there is
exactly one real equilibrium for every $h$; the other two roots of
$x^3=h$ are complex and not relevant here).

Differentiating, $f'(x) = -3x^2$, so $$
f'\!\left(x_{eq}\right) = -3h^{2/3} \leq 0 \quad \text{for every } h,
$$ with equality only at $h=0$. Thus $x_{eq}(h) = h^{1/3}$ is **stable
for every $h \neq 0$**. At $h=0$ (so $x_{eq}=0$), Theorem 1.32 is
inconclusive since $f'(0)=0$, but checking $f(x,0) = -x^3$ directly
shows $f>0$ for $x<0$ and $f<0$ for $x>0$ — trajectories are still drawn
toward $0$ from both sides, so $x=0$ remains stable (just
non-hyperbolically so).

> **Note**
>
> There is no actual bifurcation here: a single, always-stable branch of
> equilibria exists for every value of $h$, and no qualitative change in
> the equilibrium structure ever occurs. This example is a useful
> contrast to (b)–(d) below, where genuine bifurcations do occur.

In [13]:
h = np.linspace(-4, 4, 400)
x_eq = np.sign(h) * np.abs(h)**(1/3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h, x_eq, color='seagreen', lw=2.5, label='stable')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$')
ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = h - x^3$")
ax.legend()
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

### Exercise 1(b): $x' = (x-1)(x-h)$

**By Hand.**

The equilibria are $x=1$ and $x=h$. Differentiating,
$f'(x) = (x-h)+(x-1) = 2x-(1+h)$, so $$
f'(1) = 1-h, \qquad f'(h) = h-1.
$$

- $x=1$ is **stable** when $h>1$ (so $f'(1)<0$) and **unstable** when
  $h<1$.
- $x=h$ is **stable** when $h<1$ (so $f'(h)<0$) and **unstable** when
  $h>1$.

At $h=1$, the two equilibria collide at $x=1$ and *exchange stability*
as $h$ increases through $1$ — this is the hallmark of a **transcritical
bifurcation**.

In [14]:
h_sym, x_sym = sym.symbols('h x')
f = (x_sym-1)*(x_sym-h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, h]
  f'(1) = 1 - h
  f'(h) = h - 1

In [15]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 1], h[h >= 1]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.ones_like(h_lo), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, np.ones_like(h_hi), color='seagreen', lw=2.5)
ax.plot(h_lo, h_lo, color='seagreen', lw=2.5)
ax.plot(h_hi, h_hi, color='crimson', lw=2.5, linestyle='--')
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (x-1)(x-h)$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

### Exercise 1(c): $y' = hy - y^2$

**By Hand.**

Factoring, $f(y) = y(h-y) = 0$ gives equilibria $y=0$ and $y=h$.
Differentiating, $f'(y) = h-2y$, so $$
f'(0) = h, \qquad f'(h) = -h.
$$

- $y=0$ is **stable** for $h<0$ and **unstable** for $h>0$.
- $y=h$ is **unstable** for $h<0$ and **stable** for $h>0$.

At $h=0$ the two equilibria merge at $y=0$ and swap stability — this is
the textbook normal form for a **transcritical bifurcation**.

In [16]:
h_sym, y_sym = sym.symbols('h y')
f = h_sym*y_sym - y_sym**2
eqs = sym.solve(sym.Eq(f,0), y_sym)
fp = sym.diff(f, y_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(y_sym, e)))

Equilibria: [0, h]
  f'(0) = h
  f'(h) = -h

In [17]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 0], h[h >= 0]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.zeros_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.zeros_like(h_hi), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_lo, h_lo, color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, h_hi, color='seagreen', lw=2.5)
ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$y_{eq}$')
ax.set_title(r"Bifurcation diagram: $y' = hy - y^2$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

### Exercise 1(d): $x' = (1-x)(x^2-h)$

**By Hand.**

Setting $f(x) = (1-x)(x^2-h) = 0$ gives $x=1$ (for every $h$) together
with $x = \pm\sqrt{h}$ whenever $h \geq 0$ (no real square-root
equilibria when $h<0$). Differentiating, $$
f'(x) = -(x^2-h) + (1-x)(2x) = h + 2x - 3x^2,
$$ so $$
f'(1) = h - 1, \qquad f'(\sqrt{h}) = 2\sqrt{h}\left(1-\sqrt{h}\right), \qquad f'(-\sqrt{h}) = -2\sqrt{h}\left(1+\sqrt{h}\right).
$$

- $x=1$: **stable** for $h<1$, **unstable** for $h>1$.
- $x=-\sqrt{h}$ (exists for $h \geq 0$): since $\sqrt{h}\geq0$,
  $f'(-\sqrt h)\leq 0$ always $\Rightarrow$ **always stable**.
- $x=\sqrt{h}$ (exists for $h \geq 0$):
  $f'(\sqrt h) = 2\sqrt h(1-\sqrt h)$ is positive for $0<h<1$
  (**unstable**) and negative for $h>1$ (**stable**).

Two qualitative changes occur as $h$ varies:

- **At $h=0$:** for $h<0$ there is only the single equilibrium $x=1$; as
  $h$ increases through $0$, a pair of equilibria $x=\pm\sqrt h$ is born
  out of the degenerate double root at $x=0$ (where, at $h=0$,
  $f(x,0)=(1-x)x^2$ is semistable: attracting from the left, repelling
  from the right). This is a **saddle-node bifurcation**.
- **At $h=1$:** the branches $x=1$ and $x=\sqrt h$ meet (both equal $1$
  there) and exchange stability — a **transcritical bifurcation**,
  exactly as in parts (b) and (c).

In [18]:
h_sym, x_sym = sym.symbols('h x')
f = (1-x_sym)*(x_sym**2 - h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, -sqrt(h), sqrt(h)]
  f'(1) = h - 1
  f'(-sqrt(h)) = -2*sqrt(h) - 2*h
  f'(sqrt(h)) = 2*sqrt(h) - 2*h

In [19]:
h_pos = np.linspace(0, 3, 400)
h_all = np.linspace(-3, 3, 400)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

h_lo, h_hi = h_all[h_all <= 1], h_all[h_all >= 1]
ax.plot(h_lo, np.ones_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.ones_like(h_hi), color='crimson', lw=2.5, linestyle='--')

h1, h2 = h_pos[h_pos <= 1], h_pos[h_pos >= 1]
ax.plot(h1, np.sqrt(h1), color='crimson', lw=2.5, linestyle='--')
ax.plot(h2, np.sqrt(h2), color='seagreen', lw=2.5)
ax.plot(h_pos, -np.sqrt(h_pos), color='seagreen', lw=2.5)

ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (1-x)(x^2-h)$", pad=15)
ax.set_ylim(-1.9, 2.3)
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles, loc='upper left')
ax.annotate('saddle-node\n(h=0)', xy=(0, 0), xytext=(-2.7, -1.6), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
ax.annotate('transcritical\n(h=1)', xy=(1, 1), xytext=(1.7, 1.75), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
plt.tight_layout()
plt.show()

## References

> **Expand for Session Info**
>
> ``` python
> import sys # sys for system-specific parameters and functions
> print("Python version:", sys.version)
> print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m, '__version__', None)))
> ```
>
>     Python version: 3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 02:33:53) [Clang 20.1.8 ]
>     numpy==2.4.3
>     scipy==1.17.1
>     sympy==1.14.0
>     matplotlib==3.10.8

## Reuse

[![](http://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.png?raw=1)](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode)

[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)